In [17]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
# If using Target Encoding for high-cardinality columns like 'brand':
# from category_encoders import TargetEncoder

In [18]:
data = pd.read_csv('used_cars.csv')


In [19]:
data['clean_title'] = data['clean_title'].eq('Yes').astype(int)

In [20]:
data.sample(5)

,id,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
126039,126039,Jeep,Wrangler Sport,2002,116157,Gasoline,181.0HP 4.0L Straight 6 Cylinder Engine Gasoli...,A/T,Black,Gray,At least 1 accident or damage reported,1,12250
119097,119097,Ford,Bronco Outer Banks Advanced,2023,11806,Gasoline,275.0HP 2.3L 4 Cylinder Engine Gasoline Fuel,A/T,White,Brown,None reported,1,132900
24644,24644,Mercedes-Benz,E-Class E 350 4MATIC,2014,84450,Gasoline,302.0HP 3.5L V6 Cylinder Engine Gasoline Fuel,7-Speed A/T,Black,Black,At least 1 accident or damage reported,1,17000
56530,56530,Nissan,GT-R Premium,2020,8400,Gasoline,565.0HP 3.8L V6 Cylinder Engine Gasoline Fuel,Transmission w/Dual Shift Mode,White,Black,None reported,1,80000
94381,94381,BMW,540 i xDrive,2020,38356,Gasoline,2.0 Liter Turbo,Automatic,Black Sapphire Metallic,Black,None reported,0,44998


In [21]:
data = data.drop(columns='id')

In [22]:
data.head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
0,MINI,Cooper S Base,2007,213000,Gasoline,172.0HP 1.6L 4 Cylinder Engine Gasoline Fuel,A/T,Yellow,Gray,None reported,1,4200
1,Lincoln,LS V8,2002,143250,Gasoline,252.0HP 3.9L 8 Cylinder Engine Gasoline Fuel,A/T,Silver,Beige,At least 1 accident or damage reported,1,4999
2,Chevrolet,Silverado 2500 LT,2002,136731,E85 Flex Fuel,320.0HP 5.3L 8 Cylinder Engine Flex Fuel Capab...,A/T,Blue,Gray,None reported,1,13900
3,Genesis,G90 5.0 Ultimate,2017,19500,Gasoline,420.0HP 5.0L 8 Cylinder Engine Gasoline Fuel,Transmission w/Dual Shift Mode,Black,Black,None reported,1,45000
4,Mercedes-Benz,Metris Base,2021,7388,Gasoline,208.0HP 2.0L 4 Cylinder Engine Gasoline Fuel,7-Speed A/T,Black,Beige,None reported,1,97500


In [23]:
X = data.drop(columns='price')
y = np.log1p(data['price'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [24]:
def extract_features(X_in):
    X_out = X_in.copy()

    # calc vehicle age (current year is 2026)
    if 'model_year' in X_out.columns:
        X_out['vehicle_age'] = 2026 - X_out['model_year']
        X_out = X_out.drop(columns=['model_year'])

    # extract horsepower from engine string
    if 'engine' in X_out.columns:
        X_out['horsepower'] = X_out['engine'].str.extract(r'(\d+\.?\d*)HP').astype(float)
        X_out = X_out.drop(columns=['engine'])

    return X_out

# wrap the function for pipeline
feature_engineering_step = FunctionTransformer(extract_features)

# test it on X_train
X_train_engineered = feature_engineering_step.fit_transform(X_train)
X_train_engineered.head()

,brand,model,milage,fuel_type,transmission,ext_col,int_col,accident,clean_title,vehicle_age,horsepower
184031,Mercedes-Benz,GLA-Class GLA 250 4MATIC,61675,E85 Flex Fuel,Transmission w/Dual Shift Mode,Gray,Black,At least 1 accident or damage reported,1,9,208.0
173831,Nissan,Frontier SV,185000,Gasoline,A/T,Silver,Gray,At least 1 accident or damage reported,1,23,239.0
183819,Ford,F-150 XLT,92000,Gasoline,10-Speed A/T,Gray,Gray,At least 1 accident or damage reported,1,6,325.0
85525,Hyundai,Elantra N Base,5483,Gasoline,6-Speed M/T,Black,Black,None reported,0,3,276.0
41872,Dodge,Charger SRT Hellcat Widebody,5000,Gasoline,6-Speed A/T,Red,Black,None reported,1,3,717.0


In [25]:
from sklearn.preprocessing import TargetEncoder

In [26]:
num_features = ['milage', 'vehicle_age', 'horsepower']
high_cat_features = ['brand', 'model', 'ext_col', 'int_col']
low_cat_features = ['transmission', 'fuel_type', 'accident']

In [27]:
# Column Transformer

#1 numerical pipeline (impute + scale)
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scale', StandardScaler())
])

#2 high cardinality pipeline (impute + target encode)
high_cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', TargetEncoder(target_type='continuous'))
])

#3 low cardinality pipeline (impute + capped onehot encode)
low_cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(max_categories=10, handle_unknown='ignore', sparse_output=False))
])

# combine them all
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_features),
        ('high_cat', high_cat_transformer, high_cat_features),
        ('low_cat', low_cat_transformer, low_cat_features)
    ]
)

In [28]:
final_pipeline = Pipeline(steps=[
    ('feature_extraction', feature_engineering_step),
    ('preprocessor', preprocessor)
])

# note: target encoding requires passing y_train during fit stage
X_train_processed = final_pipeline.fit_transform(X_train, y_train)

# transform test data (y_test is not passed to prevent data leakage)
X_test_processed = final_pipeline.transform(X_test)

print(f"Original X_train shape: {X_train.shape}")
print(f"Processed X_train shape: {X_train_processed.shape}")

Original X_train shape: (150826, 11)
Processed X_train shape: (150826, 26)


In [29]:
X_train_processed.sample()

AttributeError: 'numpy.ndarray' object has no attribute 'sample'